# Milestone 2: Main Insight
## Renewable Energy Transition Across Countries

**Course:** Visual Communication and Data Storytelling  
**Team:** Aya Abdine, Meriam El Askri, Mcolisi Austine, Ghofran Mahmoud

This notebook defines the audience, examines four candidate insights with small supporting tables, and prioritizes the central idea of the story. It uses only the cleaned project dataset. All table values and numerical observations are computed in Python. No visualizations are created.


## Target Audience

Our target audience is a general, non-expert audience interested in sustainability, renewable energy, and global development. The story is designed to make differences between national energy transitions understandable without requiring technical knowledge of energy systems.


## Data and comparison rules

The input is `data/energy_10countries_clean.csv`. Renewable-energy adoption means `renewables_share_energy`: renewables as a percentage of primary energy consumption, using OWID's legacy substitution method. Source pathways use `solar_share_elec`, `wind_share_elec`, and `hydro_share_elec`: percentages of electricity generated. **These denominators differ and must not be combined.**

GDP is total output in international dollars at 2011 prices, adjusted for inflation and differences in living costs. Population is measured in people. Missing values stay missing; no imputation or source data changes are performed. Each table explicitly identifies its observation year.

The requested OWID source is a legacy release that is no longer updated; “latest” here refers to this saved dataset. See the repository's `ENERGY_DATA_NOTES.md` for preparation details.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

# Support running with either the repository root or notebooks/ as the working directory.
ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "energy_10countries_clean.csv").is_file():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "energy_10countries_clean.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError("Run this notebook from the repository root or notebooks/.")

df = pd.read_csv(DATA_PATH)
COUNTRIES = [
    "Germany", "Denmark", "Norway", "France", "China",
    "United States", "India", "Brazil", "Japan", "South Africa",
]
SHARE = "renewables_share_energy"
MIX = ["solar_share_elec", "wind_share_elec", "hydro_share_elec"]
required = ["country", "year", "population", "gdp", SHARE, *MIX]
assert set(required).issubset(df.columns), "A required column is missing."
assert set(df["country"]) == set(COUNTRIES), "Unexpected country coverage."
assert not df.duplicated(["country", "year"]).any(), "Duplicate country-year keys."
assert df["year"].ge(2000).all(), "Unexpected observations before 2000."

def latest_complete_rows(columns):
    """Latest observed row per country, requiring all specified measures."""
    observed = df.dropna(subset=columns)
    latest = (
        observed.sort_values(["country", "year"])
        .groupby("country", sort=True).tail(1)
        .copy()
    )
    assert set(latest["country"]) == set(COUNTRIES), "A country has no complete row."
    return latest

display(Markdown(
    f"Loaded **{len(df):,} country-year observations** for "
    f"**{df['country'].nunique()} countries**, spanning "
    f"**{df['year'].min()}–{df['year'].max()}**. "
    "All country-year keys are unique."
))


Loaded **260 country-year observations** for **10 countries**, spanning **2000–2025**. All country-year keys are unique.

## Main Insight 1

**Renewable-energy adoption differs greatly across countries.**

Some countries already have a much higher renewable-energy share than others, showing that the transition is progressing very unevenly across the selected countries.

The table ranks each country's latest observed renewable share of **primary energy**, rather than its electricity share. The year is retained so that observations cannot be mistaken for a common year if coverage changes.


In [2]:
adoption = latest_complete_rows([SHARE]).sort_values(SHARE, ascending=False)
adoption_table = adoption[["country", "year", SHARE]].rename(columns={
    "country": "Country", "year": "Year",
    SHARE: "Renewables share of primary energy (%)",
}).reset_index(drop=True)

display(adoption_table.style.hide(axis="index").format({
    "Year": "{:.0f}", "Renewables share of primary energy (%)": "{:.2f}",
}))
leader, lowest = adoption.iloc[0], adoption.iloc[-1]
gap = leader[SHARE] - lowest[SHARE]
display(Markdown(
    f"**Evidence:** Latest observations span "
    f"**{adoption['year'].min()}–{adoption['year'].max()}**. "
    f"{leader['country']} has the highest share (**{leader[SHARE]:.2f}%**) "
    f"and {lowest['country']} the lowest (**{lowest[SHARE]:.2f}%**), "
    f"a gap of **{gap:.2f} percentage points**. "
    "This supports a large difference in adoption levels within this selected group."
))


Country,Year,Renewables share of primary energy (%)
Norway,2024,71.81
Brazil,2024,49.62
Denmark,2024,41.72
Germany,2024,23.97
China,2024,17.47
France,2024,16.46
Japan,2024,12.70
United States,2024,12.05
India,2024,9.15
South Africa,2024,3.56


**Evidence:** Latest observations span **2024–2024**. Norway has the highest share (**71.81%**) and South Africa the lowest (**3.56%**), a gap of **68.25 percentage points**. This supports a large difference in adoption levels within this selected group.

## Main Insight 2

**Countries follow different renewable-energy pathways.**

Countries with high renewable-energy shares do not necessarily rely on the same sources. Some depend more on hydropower, while others rely more heavily on wind or solar.

The table uses the latest year with all three electricity-source shares available for each country. These are shares of **all electricity generated**, not percentages within renewables. Bioenergy and other renewable sources are not included in these three columns.


In [3]:
pathways = latest_complete_rows(MIX).sort_values("country")
pathways_table = pathways[["country", "year", *MIX]].rename(columns={
    "country": "Country", "year": "Year",
    "solar_share_elec": "Solar share of electricity (%)",
    "wind_share_elec": "Wind share of electricity (%)",
    "hydro_share_elec": "Hydro share of electricity (%)",
}).reset_index(drop=True)

display(pathways_table.style.hide(axis="index").format({
    "Year": "{:.0f}", **{c: "{:.2f}" for c in pathways_table.columns[2:]},
}))
hydro_leader = pathways.loc[pathways["hydro_share_elec"].idxmax()]
wind_leader = pathways.loc[pathways["wind_share_elec"].idxmax()]
display(Markdown(
    f"**Evidence:** Latest complete source-mix observations span "
    f"**{pathways['year'].min()}–{pathways['year'].max()}**. "
    f"{hydro_leader['country']} generates **{hydro_leader['hydro_share_elec']:.2f}%** "
    f"of its electricity from hydropower, while {wind_leader['country']} generates "
    f"**{wind_leader['wind_share_elec']:.2f}%** from wind. "
    "This supports different electricity pathways. The clearest contrast is hydro "
    "versus wind; the table does not imply that solar dominates a country's entire "
    "renewable mix. This electricity evidence complements, but does not decompose, "
    "the primary-energy shares in Insight 1."
))


Country,Year,Solar share of electricity (%),Wind share of electricity (%),Hydro share of electricity (%)
Brazil,2025,11.81,15.67,51.78
China,2025,11.10,10.73,13.20
Denmark,2025,13.36,57.69,0.06
France,2025,5.58,8.16,10.42
Germany,2025,17.91,27.18,3.91
India,2025,9.41,5.00,8.55
Japan,2025,9.81,1.25,7.18
Norway,2025,0.33,8.43,90.03
South Africa,2025,8.02,4.66,0.75
United States,2025,8.60,10.28,5.35


**Evidence:** Latest complete source-mix observations span **2025–2025**. Norway generates **90.03%** of its electricity from hydropower, while Denmark generates **57.69%** from wind. This supports different electricity pathways. The clearest contrast is hydro versus wind; the table does not imply that solar dominates a country's entire renewable mix. This electricity evidence complements, but does not decompose, the primary-energy shares in Insight 1.

## Main Insight 3

**Economic size does not automatically determine renewable-energy leadership.**

Large economies or highly populated countries are not necessarily the countries with the highest renewable-energy shares.

For a fair comparison, the table uses the latest **common year** with GDP, population, and primary-energy renewable share observed for **all ten countries**. It sorts by total GDP so that economic size can be compared directly with renewable share. Population and GDP values are displayed without changing their source units.


In [4]:
context_columns = ["gdp", "population", SHARE]
complete_context = df.dropna(subset=context_columns)
common_years = (
    complete_context.groupby("year")["country"].nunique()
    .loc[lambda counts: counts.eq(len(COUNTRIES))].index
)
if len(common_years) == 0:
    raise ValueError("No common complete year exists for the economic comparison.")
context_year = int(common_years.max())
context = (
    complete_context.loc[complete_context["year"].eq(context_year)]
    .sort_values("gdp", ascending=False).copy()
)
assert len(context) == len(COUNTRIES)
context_table = context[["country", "gdp", "population", SHARE]].rename(columns={
    "country": "Country", "gdp": "GDP (2011 international $)",
    "population": "Population (people)",
    SHARE: "Renewables share of primary energy (%)",
}).reset_index(drop=True)

display(Markdown(f"**Common comparison year: {context_year}.**"))
display(context_table.style.hide(axis="index").format({
    "GDP (2011 international $)": "{:,.0f}",
    "Population (people)": "{:,.0f}",
    "Renewables share of primary energy (%)": "{:.2f}",
}))
largest_economy = context.iloc[0]
largest_population = context.loc[context["population"].idxmax()]
renewable_leader = context.loc[context[SHARE].idxmax()]
display(Markdown(
    f"**Evidence:** In **{context_year}**, {largest_economy['country']} has the "
    f"largest total GDP in the sample and a renewable share of "
    f"**{largest_economy[SHARE]:.2f}%**. "
    f"{largest_population['country']} has the largest population and a share of "
    f"**{largest_population[SHARE]:.2f}%**. "
    f"The renewable-share leader is {renewable_leader['country']} "
    f"(**{renewable_leader[SHARE]:.2f}%**). "
    "These comparisons support the claim that size alone does not establish "
    "leadership by renewable share. They do not establish causation or prove "
    "that economic size and renewable adoption are unrelated. Leadership here "
    "means share, not absolute renewable generation."
))


**Common comparison year: 2022.**

Country,GDP (2011 international $),Population (people),Renewables share of primary energy (%)
China,"26,966,016,991,166","1,425,179,562",15.22
United States,"19,493,170,182,843","341,534,041",10.88
India,"10,476,248,645,535","1,425,423,209",9.42
Japan,"4,774,494,846,664","124,997,585",11.15
Germany,"3,909,612,753,939","84,086,227",20.33
Brazil,"3,187,412,907,763","210,306,411",47.66
France,"2,671,568,767,031","66,277,412",13.73
South Africa,"723,559,323,969","62,378,417",3.65
Norway,"482,328,451,592","5,456,796",69.65
Denmark,"299,429,011,832","5,902,901",40.79


**Evidence:** In **2022**, China has the largest total GDP in the sample and a renewable share of **15.22%**. India has the largest population and a share of **9.42%**. The renewable-share leader is Norway (**69.65%**). These comparisons support the claim that size alone does not establish leadership by renewable share. They do not establish causation or prove that economic size and renewable adoption are unrelated. Leadership here means share, not absolute renewable generation.

## Main Insight 4

**Countries have made very different gains in renewable-energy share since 2000.**

The endpoint comparison shows that countries have experienced very different cumulative changes in renewable-energy share since 2000.

The table compares the exact 2000 observation with each country's latest observed primary-energy renewable share. It does not substitute a nearby year if 2000 is unavailable.

**Reading the change columns:** absolute change means the signed difference, latest share minus starting share, rather than relative percentage growth. Because the source values are percentages, this difference is already a **percentage-point change**. Both requested columns show the same quantity with explicit labels; they are not separate measures. A positive value indicates an increase.

Endpoint changes show cumulative progress over the period. They do not establish a steady annual pace or describe fluctuations between endpoints. Countries starting with high renewable shares also have less room for percentage-point increases.


In [5]:
baseline_year = 2000
baseline = df.loc[df["year"].eq(baseline_year), ["country", SHARE]].rename(
    columns={SHARE: "start_share"}
)
assert set(baseline["country"]) == set(COUNTRIES)
assert baseline["start_share"].notna().all(), "A baseline renewable share is missing."
endpoints = latest_complete_rows([SHARE])[["country", "year", SHARE]].rename(
    columns={"year": "latest_year", SHARE: "latest_share"}
)
change = baseline.merge(endpoints, on="country", validate="one_to_one")
change["absolute_change"] = change["latest_share"] - change["start_share"]
change["percentage_point_change"] = change["absolute_change"]
change = change.sort_values("percentage_point_change", ascending=False)
change_table = change.assign(start_year=baseline_year)[[
    "country", "start_year", "start_share", "latest_year", "latest_share",
    "absolute_change", "percentage_point_change",
]].rename(columns={
    "country": "Country", "start_year": "Start year",
    "start_share": "Starting renewable share (%)",
    "latest_year": "Latest year", "latest_share": "Latest renewable share (%)",
    "absolute_change": "Absolute change (share units)",
    "percentage_point_change": "Change (percentage points)",
}).reset_index(drop=True)

display(change_table.style.hide(axis="index").format({
    "Start year": "{:.0f}", "Latest year": "{:.0f}",
    "Starting renewable share (%)": "{:.2f}",
    "Latest renewable share (%)": "{:.2f}",
    "Absolute change (share units)": "{:+.2f}",
    "Change (percentage points)": "{:+.2f}",
}))
largest_gain, smallest_gain = change.iloc[0], change.iloc[-1]
display(Markdown(
    f"**Evidence:** The latest endpoint spans "
    f"**{change['latest_year'].min()}–{change['latest_year'].max()}**. "
    f"{largest_gain['country']} records the largest increase since {baseline_year} "
    f"(**{largest_gain['percentage_point_change']:+.2f} percentage points**), "
    f"while {smallest_gain['country']} records the smallest "
    f"(**{smallest_gain['percentage_point_change']:+.2f} percentage points**). "
    "The cumulative increases differ markedly. "
    "**Suggested precise wording:** Countries have made very different "
    "percentage-point gains in renewable-energy share since 2000. "
    "An endpoint comparison alone cannot establish acceleration, consistency, "
    "or the timing of change."
))


Country,Start year,Starting renewable share (%),Latest year,Latest renewable share (%),Absolute change (share units),Change (percentage points)
Denmark,2000,6.76,2024,41.72,+34.96,+34.96
Germany,2000,2.74,2024,23.97,+21.23,+21.23
China,2000,5.34,2024,17.47,+12.14,+12.14
France,2000,6.41,2024,16.46,+10.06,+10.06
Brazil,2000,40.94,2024,49.62,+8.68,+8.68
United States,2000,3.86,2024,12.05,+8.19,+8.19
Japan,2000,4.60,2024,12.70,+8.09,+8.09
South Africa,2000,0.39,2024,3.56,+3.17,+3.17
India,2000,6.04,2024,9.15,+3.10,+3.10
Norway,2000,70.79,2024,71.81,+1.02,+1.02


**Evidence:** The latest endpoint spans **2024–2024**. Denmark records the largest increase since 2000 (**+34.96 percentage points**), while Norway records the smallest (**+1.02 percentage points**). The cumulative increases differ markedly. **Suggested precise wording:** Countries have made very different percentage-point gains in renewable-energy share since 2000. An endpoint comparison alone cannot establish acceleration, consistency, or the timing of change.

## Main Prioritized Insight

**There is no single path to renewable energy. Countries differ in how much renewable energy they use, which renewable sources they rely on, and how their renewable-energy shares have changed over time.**

This will be the central idea guiding the final data story. The supporting evidence connects differences in adoption levels, electricity-source mixes, and cumulative change over time. Economic and demographic context helps frame those differences without implying a single cause.

## Assessment of the four insights

1. **Adoption differences — supported.** The latest primary-energy renewable shares differ substantially across the selected countries.
2. **Different pathways — supported with a scope clarification.** Hydro and wind provide clear contrasting electricity pathways. Electricity-source shares should not be presented as a decomposition of total primary energy, and the evidence does not require claiming solar dominance.
3. **Economic size — supported as a descriptive statement.** The largest economies and populations are not the renewable-share leaders. This small, deliberately selected sample cannot establish causation or the absence of a statistical relationship.
4. **Different transition pace — supported as differing cumulative gains.** For this endpoint table, “Countries have made very different percentage-point gains in renewable-energy share since 2000” is more precise than a claim about steady speed or acceleration. Starting levels also matter.

All conclusions apply to these selected countries and the observed periods. Recent GDP gaps require an earlier common year for the economic comparison. No work on later milestones, charts, or the 3D idea is included.
